In [1]:
# from typing import Dict, TypedDict, List, Annotated, Sequence
from langgraph.graph import StateGraph, START, END
# from langgraph.prebuilt import ToolNode
# from langgraph.graph.message import add_messages
# 
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# from langchain_core.tools import tool
# from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage, BaseMessage
from typing import Optional, Any
from typing_extensions import TypedDict

from pydantic import BaseModel, Field

import instructor
from openai import OpenAI

from dotenv import load_dotenv



In [2]:
load_dotenv()

True

In [3]:
# -------------------------
# 1. Structured output schema
# -------------------------

class Actor(BaseModel):
    name: str = Field(..., description="Actor name exactly as referred to in the text")
    actor_type: str = Field(
        ...,
        description="Actor type, e.g. Person, Group, System"
    )


class WorkObject(BaseModel):
    name: str = Field(
        ...,
        description="Exact work object name mentioned in the text."
    )
    object_type: str = Field(
        ...,
        description=(
            "General normalized category of the work object, such as document, form, "
            "data, email, request, report, message, contract, file, record, "
            "payment, phone call, or conversation."
        )
    )
    description: str = Field(
        ...,
        description=(
            "Short, general semantic description of the work object for icon retrieval. "
            "Describe only what kind of thing it is in general, not the specific story context. "
            "Focus on broad visual meaning such as form, document, report, data, message, "
            "ticket, phone call, or digital record. The description may refer either to the "
            "work object itself or to the medium through which information about it is exchanged."
        )
    )


class Activity(BaseModel):
    step: int = Field(..., description="Activity step number")
    actor: str = Field(..., description="Actor performing the activity")
    verb: str = Field(..., description="Verb describing the activity")
    work_object: str = Field(..., description="Work object involved in the activity")
    preposition: Optional[str] = Field(None, description="Optional preposition for the activity")
    involved_actor: Optional[str] = Field(None, description="Optional second actor involved in the activity")


class DomainStoryElements(BaseModel):
    actors: list[Actor] = Field(default_factory=list)
    work_objects: list[WorkObject] = Field(default_factory=list)
    activities: list[Activity] = Field(default_factory=list)


# -------------------------
# 2. LangGraph state
# -------------------------

class AgentState(TypedDict):
    text: str
    extracted_elements: Optional[DomainStoryElements]
    icon_matches: Optional[dict[str, dict[str, Any]]]
    error: Optional[str]


# -------------------------
# 3. Model client
# -------------------------

client = instructor.from_openai(OpenAI())

system_prompt = """
You extract Domain Story elements from the given text.

For each work object, return:
- the exact name from the text
- a general object_type
- a short description for icon retrieval

Rules for the description:
- Keep it general and context-independent
- Do not describe what happens to the object in the story
- Do not mention specific actors, systems, or actions
- Focus on the broad semantic or visual meaning of the object
- The description may refer either to the object itself or to the medium through which it is exchanged
- Use short noun phrases such as:
  'document, form, paperwork'
  'information, digital data, record'
  'email, message, communication'
  'ticket, pass, admission document'
  'phone call, spoken communication'
"""


# -------------------------
# 4. Extraction node
# -------------------------

def extraction_node(state: AgentState) -> AgentState:
    try:
        result = client.chat.completions.create(
            model="gpt-4o-mini",
            response_model=DomainStoryElements,
            temperature=0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": state["text"]},
            ],
        )

        return {
            "extracted_elements": result,
            "error": None,
        }

    except Exception as e:
        return {
            "extracted_elements": None,
            "error": str(e),
        }




In [4]:
# print(result["work_objects"].model_dump())

## Semantic Search for Icons

In [5]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

ICON_FILE = r"C:\code\NL_2_DST\material_icons\semantics_added.json"
CHROMA_DIR = r"C:\code\NL_2_DST\material_icons\chroma_mdi_icons"
COLLECTION_NAME = "mdi_icons"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def build_icon_documents(icon_file: str) -> list[Document]:
    icons = json.loads(Path(icon_file).read_text(encoding="utf-8"))
    docs = []

    for icon in icons:
        semantic = icon.get("semanticMeaning", "")
        aliases = ", ".join(icon.get("aliases", []))
        tags = ", ".join(icon.get("tags", []))

        page_content = (
            f"name: {icon.get('name', '')}\n"
            f"aliases: {aliases}\n"
            f"tags: {tags}\n"
            f"semantic meaning: {semantic}"
        )

        metadata = {
            "name": icon.get("name", ""),
            "jsExportName": icon.get("jsExportName", ""),
            "svgPath": icon.get("svgPath", ""),
            "aliases": ", ".join(icon.get("aliases", [])),
            "tags": ", ".join(icon.get("tags", [])),
            "semanticMeaning": semantic,
        }

        docs.append(Document(page_content=page_content, metadata=metadata))

    return docs

# 
def get_or_build_icon_store() -> Chroma:
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    )

    existing = vector_store.get(limit=1)
    if not existing["ids"]:
        docs = build_icon_documents(ICON_FILE)

        # keep this comfortably below the max batch size
        for batch in chunked(docs, 200):
            vector_store.add_documents(batch)

    return vector_store
# 
# 


C:\code\NL_2_DST\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def build_icon_query(work_object: WorkObject) -> str:
    return (
        f"{work_object.object_type}. "
        f"{work_object.description}. "
        f"{work_object.name}"
    )
vector_store = get_or_build_icon_store()


def icon_search_node(state: AgentState) -> AgentState:
    try:
        extracted = state.get("extracted_elements")
        if not extracted:
            return {
                "icon_matches": {},
                "error": "No extracted elements available."
            }

        matches = {}

        for work_object in extracted.work_objects:
            query = build_icon_query(work_object)

            results = vector_store.similarity_search_with_score(query, k=1)

            if results:
                doc, score = results[0]

                matches[work_object.name] = {
                    "query": query,
                    "matched_icon_name": doc.metadata.get("name"),
                    "jsExportName": doc.metadata.get("jsExportName"),
                    "svgPath": doc.metadata.get("svgPath"),
                    "semanticMeaning": doc.metadata.get("semanticMeaning"),
                    "score": score,
                }
            else:
                matches[work_object.name] = {
                    "query": query,
                    "matched_icon_name": None,
                    "jsExportName": None,
                    "svgPath": None,
                    "semanticMeaning": None,
                    "score": None,
                }

        return {
            "icon_matches": matches,
            "error": None,
        }

    except Exception as e:
        return {
            "icon_matches": {},
            "error": str(e),
        }

In [7]:
# -------------------------
# 5. Build graph
# -------------------------

graph = StateGraph(AgentState)

graph.add_node("extraction_node", extraction_node)
graph.add_node("icon_search_node", icon_search_node)

graph.add_edge(START, "extraction_node")
graph.add_edge("extraction_node", "icon_search_node")
graph.add_edge("icon_search_node", END)

agent = graph.compile()

# -------------------------
# 6. Example run
# -------------------------
alphorn_1_path = r"C:\code\NL_2_DST\alphorn_text\alphorn-1-standardcase.txt"


def load_content(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


content = load_content(alphorn_1_path)

result = agent.invoke(
    {
        "text": content
    }
)
# result = agent.invoke(
#     {
#         "text": "The customer sends an order form to the sales department. "
#                 "The sales employee checks the form and enters the data into the ERP system."
#     }
# )

print(result)



{'text': '1.The customer chooses a specific car from the catalog.\n\n2.The customer ask the salesperson for an offer.\n\n3.The salesperson offers a contract for a car with a monthly installment to the customer.\n\n4.The customer signs the contract with the salesperson.\n\n5.The salesperson passes the contract on to the risk manager.\n\n6.The risk manager assesses the risk for the contract.\n\n7.The risk manager votes on the contract.\n\n8.The risk manager informs the salesperson about the voting result.\n\n9.The salesperson hands over the car to the customer.\n\n10.The customer pays the monthly installment regularly\n\n11.The customer returns the car to the salesperson after the leasing period.', 'extracted_elements': DomainStoryElements(actors=[Actor(name='customer', actor_type='Person'), Actor(name='salesperson', actor_type='Person'), Actor(name='risk manager', actor_type='Person')], work_objects=[WorkObject(name='car', object_type='product', description='vehicle'), WorkObject(name='

In [ ]:
icons = result["icon_matches"]
icons

In [9]:
# # Extraction Node
# def extract_elements(user_input: str):
#     """Extracts elements from the user input."""
#     # This is a placeholder implementation. You can replace it with actual logic to extract elements.
#     pass
# 
# 
# # Icon search Node
# def search_icons():
#     pass
# 
# 
# # PlantUML Generation Node
# def generate_plantuml(elements: Dict[str, List[str]]) -> str:
#     """Generates PlantUML code from the extracted elements."""
#     pass
# 
# 
# # Syntax validation Node
# def validate_plantuml(plantuml_code: str) -> bool:
#     """Validates the generated PlantUML code."""
#     pass
# 
# 
# # Conditional Node
# def is_valid(plantuml_code: str) -> bool:
#     """Checks if the PlantUML code is valid."""
#     pass
# 
# # tools = [extract_elements, generate_plantuml]

In [10]:
# llm = ChatOpenAI(model="gpt-4o")
# 
# system_prompt = """
# You extract Domain Story elements from the given text.
# Return these detected actors and work objects=
# """
# 
# 
# def model_node(state: AgentState) -> AgentState:
#     messages = list(state["messages"])
#     messages = [SystemMessage(content=system_prompt)] + messages
#     message = llm.invoke(messages)
#     return {"message": [message]}
# 
# 


In [11]:
# graph = StateGraph(AgentState)
# graph.add_node("model_node", model_node)
# graph.add_node("extraction_node", extract_elements)
# graph.add_node("icon_search_node", search_icons)
# graph.add_node("plantuml_generation_node", generate_plantuml)
# graph.add_node("validation_node", validate_plantuml)
# 
# graph.set_entry_point("model_node")
# graph.add_edge("model_node", "extraction_node")
# graph.add_edge("extraction_node", "icon_search_node")
# graph.add_edge("icon_search_node", "plantuml_generation_node")
# graph.add_edge("plantuml_generation_node", "validation_node")
# 
# graph.add_conditional_edges("validation_node", is_valid, {"valid": END, "invalid": "plantuml_generation_node"})
# agent = graph.compile()

In [12]:
# from IPython.display import display, Image
# 
# display(Image(agent.get_graph().draw_mermaid_png()))